# This is a broker metrics related forecasting analysis for the metrics:

### query/bytes 
### query/cpu/time 
### query/node/bytes 
### query/node/time 
### query/node/ttfb 
### query/priority 
### query/time

In [8]:
from pydruid.client import *
from pydruid.utils import dimensions, filters, aggregators
import pandas as pd

In [11]:
query = PyDruid('http://test0.druidonk8s.druid.data-infra.shopee.io/', 'druid/v2')
query.set_basic_auth_credentials("admin", "s2Ysn5v5")

In [12]:
ts = query.scan(
    datasource='druid_cenarius_metrics_all',
    granularity='all',
    intervals='2022-07-01/2022-08-03',
    limit=100000,
    context={"timeout": 1000},
    columns=["__time","value"],
    filter=((filters.Dimension('metric') == 'query/bytes') & (filters.Dimension('service') == 'druid/historical'))
)
df = query.export_pandas()
df = df.dropna(how='all') 
df['__time'] = pd.to_datetime(df['__time'], unit='ms').dt.tz_localize(None)
df1 = df.rename(columns={"__time":"ds","value":"y"})
display(df1)

JSONDecodeError: Expecting ',' delimiter: line 1 column 1225446 (char 1225445)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure([go.Scatter(x=df['ds'], y=df['y'])])
fig.show()

In [ ]:
ts = query.timeseries(
    datasource='druid_cenarius_metrics_all',
    granularity='hour',
    intervals='2022-07-01/2022-08-03',
    #paging_spec={'pagingIdentifies': {}, 'threshold': 1},
    #context={"timeout": 1000},
    aggregations={'maxval': aggregators.longmax('value')},
    filter=(filters.Dimension('metric') == 'query/node/bytes') & (filters.Dimension('service') == 'druid/broker')
)
df = query.export_pandas()
df['ds'] = pd.to_datetime(df['timestamp']).dt.date
df.drop(columns=['timestamp'])
df1 = df.rename(columns={"maxval":"y"})

In [ ]:
from prophet import Prophet

m = Prophet()
m.fit(df1)

In [ ]:
future = m.make_future_dataframe(periods=120)
future.tail()

In [ ]:
forecast = m.predict(future)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

In [ ]:
fig1 = m.plot(forecast)


In [ ]:
fig2 = m.plot_components(forecast)

In [ ]:
from prophet.plot import plot_plotly, plot_components_plotly

plot_plotly(m, forecast)

In [ ]:
plot_components_plotly(m, forecast)